In [1]:
# Установим библиотеки, если они ещё не установлены
# !pip install pandas numpy seaborn scikit-learn

In [2]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [3]:
# Загружаем датасет Adult (можно указать локальный путь или скачать напрямую)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

# Указываем имена колонок, так как они не заданы в файле
column_names = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

# Загрузка данных
data = pd.read_csv(url, names=column_names, na_values=" ?", skipinitialspace=True)

# Посмотрим на первые строки
data.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [4]:
# Проверим наличие пропущенных значений
data.isnull().sum()

age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64

In [5]:
# Заполним пропуски в категориальных столбцах самой частой категорией
categorical_cols = data.select_dtypes(include='object').columns.tolist()
numerical_cols = data.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Разделим признаки и целевую переменную
X = data.drop("income", axis=1)
y = data["income"]

# Создаем трансформеры
categorical_imputer = SimpleImputer(strategy="most_frequent")
numerical_imputer = SimpleImputer(strategy="mean")


In [6]:
# One-Hot кодирование категориальных признаков
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)


In [7]:
# Масштабируем числовые признаки с помощью StandardScaler
scaler = StandardScaler()


In [8]:
# Объединяем обработку категориальных и числовых признаков в ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', numerical_imputer),
            ('scaler', scaler)
        ]), numerical_cols),

        ('cat', Pipeline([
            ('imputer', categorical_imputer),
            ('encoder', encoder)
        ]), categorical_cols)
    ]
)


In [9]:
# Разделим данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Применим трансформации
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Размерность после обработки
print("Размер X_train до обработки:", X_train.shape)
print("Размер X_train после обработки:", X_train_processed.shape)


ValueError: A given column is not a column of the dataframe